# PanNuke Nuclei Detection & Phenotyping - Starter Notebook

Welcome to the **PanNuke Histopathology Nuclei Detection & Phenotyping Challenge**!

### Challenge Goals:
1. **Detect** cell nuclei in $256 \times 256$ H&E-stained tissue patches across 19 human organ types.
2. **Phenotype** each detected nucleus into one of 5 biological categories:
   - **1. Neoplastic** (Tumor cells)
   - **2. Inflammatory** (Immune infiltrating cells)
   - **3. Connective** (Stroma / Fibroblasts)
   - **4. Dead** (Apoptotic / Necrotic)
   - **5. Epithelial** (Benign epithelial cells)

This notebook will take you through loading the dataset, visualizing nuclei bounding boxes, building a Faster R-CNN detector, running inference, and generating an official submission.

In [ ]:
import os
import sys
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch.utils.data import DataLoader

# Add src and evaluation to path
sys.path.append(str(Path.cwd().parent / "src"))
sys.path.append(str(Path.cwd().parent / "evaluation"))

from dataset import PanNukeCOCODataset
from model import build_faster_rcnn_model
from utils import visualize_detections, CATEGORY_NAMES, CATEGORY_COLORS
from metrics import compute_coco_metrics
from validate_submission import validate_submission_format

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

## 1. Ensure Data is Ready
If you haven't generated the mock dataset or extracted official data yet, run the cell below:

In [ ]:
data_script = Path.cwd().parent / "data" / "download.py"
mock_dir = Path.cwd().parent / "data" / "mock_coco"

if not (mock_dir / "annotations_train.json").exists():
    print("Generating mock dataset for testing...")
    !python "{data_script}" --generate-mock --output-dir "{mock_dir}"
else:
    print(f"Dataset found at {mock_dir}")

## 2. Visualize Sample Histology Patches & Nuclei
Let's inspect training samples and view the ground-truth cell annotations.

In [ ]:
train_ds = PanNukeCOCODataset(
    images_dir=str(mock_dir / "images"),
    annotations_file=str(mock_dir / "annotations_train.json")
)
print(f"Total training patches: {len(train_ds)}")

# 1. Run full Exploratory Data Analysis (EDA) Dashboard from starter_code/visualize_data.py
sys.path.append(str(Path.cwd().parent / "starter_code"))
from visualize_data import create_pannuke_eda_dashboard

create_pannuke_eda_dashboard(
    annotations_csv=Path.cwd().parent / "data" / "public" / "train_annotations.csv",
    images_dir=mock_dir / "images",
    output_path=Path.cwd().parent / "starter_code" / "eda_pannuke_overview.png",
    show_plot=True
)

# 2. Inspect single sample with bounding boxes and phenotype labels
img_tensor, target = train_ds[0]
img_np = (img_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
img_pil = Image.fromarray(img_np)
boxes = target["boxes"].numpy().tolist()
labels = target["labels"].numpy().tolist()

print(f"Sample #0 contains {len(boxes)} annotated nuclei.")
visualize_detections(
    image=img_pil,
    boxes=boxes,
    labels=labels,
    box_format="xyxy",
    title="Ground Truth Nuclei Phenotypes (Sample #0)"
)

## 3. Initialize Faster R-CNN Detector
We load a Faster R-CNN model with a ResNet50-FPN backbone customized for our 5 cell categories (+ 1 background).

In [ ]:
model = build_faster_rcnn_model(num_classes=6, pretrained_backbone=False)
model.to(device)
print("Faster R-CNN model ready!")

## 4. Run a Training Step (Demonstration)
Let's train for 1 epoch to verify the end-to-end gradient pipeline.

In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, collate_fn=collate_fn)
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

model.train()
for step, (images, targets) in enumerate(train_loader):
    images = list(img.to(device) for img in images)
    targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
    
    loss_dict = model(images, targets)
    loss = sum(l for l in loss_dict.values())
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (step + 1) % 5 == 0:
        print(f"Step {step + 1}/{len(train_loader)} - Batch Loss: {loss.item():.4f}")
    if step >= 10:  # Quick demo stop
        break

## 5. Generate Test Predictions & Format Submission
Now let's run test inference and generate a schema-compliant submission JSON.

In [ ]:
val_ds = PanNukeCOCODataset(
    images_dir=str(mock_dir / "images"),
    annotations_file=str(mock_dir / "annotations_val.json"),
    is_test=True
)
val_loader = DataLoader(val_ds, batch_size=2, shuffle=False, collate_fn=collate_fn)

model.eval()
submission = []
conf_threshold = 0.2

with torch.no_grad():
    for images, targets in val_loader:
        images = list(img.to(device) for img in images)
        outputs = model(images)
        
        for target, output in zip(targets, outputs):
            img_id = int(target["image_id"].item())
            boxes = output["boxes"].cpu().numpy()
            scores = output["scores"].cpu().numpy()
            labels = output["labels"].cpu().numpy()
            
            keep = scores >= conf_threshold
            for b, s, l in zip(boxes[keep], scores[keep], labels[keep]):
                x1, y1, x2, y2 = b
                submission.append({
                    "image_id": img_id,
                    "category_id": int(l),
                    "bbox": [round(float(x1), 2), round(float(y1), 2), round(float(x2 - x1), 2), round(float(y2 - y1), 2)],
                    "score": round(float(s), 4)
                })

print(f"Total predicted nuclei detections: {len(submission)}")

# Save submission
sub_file = Path.cwd().parent / "sample_submission" / "notebook_submission.json"
with open(sub_file, "w", encoding="utf-8") as f:
    json.dump(submission, f, indent=2)
print(f"Saved to {sub_file}")

## 6. Validate and Score Submission

In [ ]:
# Validate
is_valid, errors = validate_submission_format(submission)
print(f"Submission Valid: {is_valid}")
if not is_valid:
    print("Errors:", errors)
else:
    # Compute benchmark metrics
    with open(mock_dir / "annotations_val.json", "r", encoding="utf-8") as f:
        val_gt = json.load(f)
    
    results = compute_coco_metrics(val_gt, submission)
    print(f"Official mAP@[0.50:0.95]: {results['mAP_50_95']:.4f}")
    print(f"Official mAP@0.50:        {results['mAP_50']:.4f}")